# InstructPix2Pix：学习遵循图像编辑说明
<!-- InstructPix2Pix: Learning to Follow Image Editing Instructions -->

[Github链接](https://github.com/timothybrooks/instruct-pix2pix)

## 摘要

该文档提出了一种名为 InstructPix2Pix 的新方法，用于根据人工指令编辑图像。该方法利用了两个大型预训练模型的能力：一个用于生成编辑指令的语言模型（GPT-3）和一个文本转图像模型（Stable Diffusion），用于创建编辑前后成对的图像数据集。通过结合这些模型，作者生成了一个庞大的数据集，用于训练条件扩散模型，能够快速准确地进行图像编辑，无需大量训练数据或用户提供的示例图像。InstructPix2Pix 模型能够直观地理解并执行一系列指令，以进行指定编辑，如修改样式、替换对象或更改设置，从而为想要编辑图像的用户提供更无缝的交互。值得注意的是，模型在合成实例上的训练并未影响其表现，使其能够有效地泛化到现实世界的图像和各种人类推理指令。这一创新技术代表了图像编辑领域的重大进展，提供了遵循用户指令进行图像转换的高效性和灵活性。


<div style="background-color:white; width:80%; margin:auto; display: flex; flex-direction: column; align-items: center; border-radius:10px">
    <img src="./assets/teaser.png" />
    <p style="text-align:center; color:black"><strong>图1：</strong>给定一张图片和编辑该图片的指令 ，我们的模型执行相应的编辑。我们的模型不需要输入或输出图像的完整描述，且在前向阶段编辑图像时无需每例反转或微调。</p>
</div>

<div style="background-color:white; width:80%; margin:auto; display: flex; flex-direction: column; align-items: center; padding: 5px; border-radius:10px">
    <img src="./assets/method.png" />
    <p style="text-align:center; color:black"><strong>图2：</strong>我们的方法分为两部分：生成图像编辑数据集，并在该数据集上训练扩散模型。（a） 我们首先使用经过微调的 GPT-3 生成指令和编辑的说明。（b）然后我们结合 StableDiffusion [52] 与Prompt-to-Prompt [17] 结合，从成对的说明生成图像对。我们利用该过程创建包含超过 45 万个训练样本的数据集（c）。（d） 最后，我们的 InstructPix2Pix 扩散模型基于生成的数据进行训练，根据指令编辑图像。在推理阶段，我们的模型推广为编辑人类指令中的真实图像。</p>
</div>

## 3 方法

我们将基于指令的图像编辑视为监督学习问题：
1) 首先生成文本编辑指令和图像的配对训练数据集（编辑前后）（第 3.1 节，图 2a-c），然后
2) 在该生成数据集上训练图像编辑扩散模型（第 3.2 节 ，图 2）d）。尽管训练时使用生成图像和编辑指令，我们的模型能够推广到使用任意人工指令编辑真实图像。见图 2，了解我们方法的概述。

### 3.1 生成多模态训练数据集

我们结合了两个在不同模态上运行的大规模预训练模型的能力——一个大型语言模型[7]和一个文本到图像模型[52]——来生成一个包含文本编辑指令以及编辑前后对应图像的多模态训练数据集。在接下来的两节中，我们将详细描述这一过程的两个步骤。在 3.1.1 节中，我们描述了微调 GPT-3[7]以生成一系列文本编辑的过程：给定一个描述图像的提示，生成一个描述要进行的更改的文本指令，以及一个描述更改后图像的提示（图 2a）。然后，在 3.1.2 节中，我们描述了使用文本到图像模型[52]将这两个文本提示（即编辑前和编辑后）转换为对应图像对的过程（图 2b）。

<style>
    /* tr样式，最后俩td用黄色 */
    .custom-tr td:last-child, .custom-tr td:nth-last-child(2) {
        font-weight: bold;
        /* color: #2c3e50; */
        color: orange;
        border: white 1px solid;
    }
</style>
<table>
    <tr style="border-top: 2px double;">
        <td> </td>
        <td> <h> INPUT LAION caption </h> </td>
        <td> <h> Edit instruction </h> </td>
        <td> <h> Edited caption </h> </td>
    </tr>
    <!-- 四行合并 -->
    <tr>
        <td rowspan="4"> Human-written (700 edits) </td>
        <td>Yefim Volkov, Misty Morning</td>
        <td>Make it a afternoon</td>
        <td>Yefim Volkov, Misty Afternoon</td>
    </tr>
    <tr>
        <td>girl with horse at sunset</td>
        <td>change the background to a city</td>
        <td>girl with horse at sunset in front of city</td>
    </tr>
    <tr>
        <td>painting-of-forest-and-pond</td>
        <td>Without the water.</td>
        <td>painting-of-forest</td>
    </tr>
    <tr>
        <td>...</td>
        <td>...</td>
        <td>...</td>
    </tr>
    <tr style="border-top: 2px double;" class="custom-tr">
        <td rowspan="4"> GPT-3 generated (450,000+ edits) </td>
        <td>Alex Hill, Original oil painting on canvas, Moonlight Bay</td>
        <td>in the style of a coloring book</td>
        <td>Alex Hill, Original coloring book illustration, Moonlight Bay</td>
    </tr>
    <tr class="custom-tr">
        <td>The great elf city of Rivendell, sitting atop a waterfall as cascades of water spill around it</td>
        <td>Add a giant red dragon</td>
        <td>The great elf city of Rivendell, sitting atop a waterfall as cascades of water spill around it with a giant red dragon flying overhead</td>
    </tr>
    <tr class="custom-tr">
        <td>Kate Hudson arriving at the Golden Globes 2015</td>
        <td>make her look like a zombie</td>
        <td>Zombie Kate Hudson arriving at the Golden Globes 2015</td>
    </tr>
    <tr style="border-bottom: 2px double;" class="custom-tr">
        <td>...</td>
        <td>...</td>
        <td>...</td>
    </tr>
    <tr>
        <td colspan="4" style="text-align:center; border-top: 2px double;">
            <strong>表1：</strong>标注了一个小文本数据集，微调GPT-3，并使用该微调模型生成一个大型文本三元组数据集。对于标注和生成的示例，我们都使用来自LAION的真实图像标题作为输入标题。高亮文本由 GPT-3 生成。
        </td>
    </tr>
</table>

<div style="background-color:white; width:80%; margin:auto; display: flex; flex-direction: column; align-items: center; padding: 5px; border-radius:10px">
    <div style="display: flex; flex-direction: row; gap: 8px; margin-bottom: 10px;">
        <div style="display: flex; flex-direction: column; align-items: center; gap: 8px;">
            <div style="display: flex; flex-direction: row; gap: 8px;">
                <img src="./assets/p2p/horse_0.jpg" style="max-width: calc(50% - 7.5px); height: auto; object-fit: contain;" />
                <img src="./assets/p2p/dragon_0.jpg" style="max-width: calc(50% - 7.5px); height: auto; object-fit: contain;" />
            </div>
            <!-- 下方添加文字 -->
            <p style="margin: 0; color: black; font-size: 14px;">(a) Without Prompt-to-Prompt</p>
        </div>
        <div style="display: flex; flex-direction: column; align-items: center; gap: 8px;">
            <div style="display: flex; flex-direction: row; gap: 8px;">
                <img src="./assets/p2p/horse_0.jpg" style="max-width: calc(50% - 7.5px); height: auto; object-fit: contain;" />
                <img src="./assets/p2p/dragon_1.jpg" style="max-width: calc(50% - 7.5px); height: auto; object-fit: contain;" />
            </div>
            <!-- 下方添加文字 -->
            <p style="margin: 0; color: black; font-size: 14px;">(b) With Prompt-to-Prompt</p>
        </div>
    </div>
    <p style="text-align:center; color:black"><strong>图3：</strong>GPT-3 生成的编辑指令和编辑后提示的示例。给定图像标题作为输入提示，GPT-3 生成描述要进行的更改的编辑指令，以及描述更改后图像的编辑后提示。</p>
</div>